# Qwen Server for DeepDive Intelligence Integration

This notebook runs a FastAPI server exposing Qwen 3 8B Instruct (quantized in 4-bit) as a public endpoint via ngrok. It connects to your local DeepDive Intelligence backend to provide LLM assessment augmentation.

## Step 1: Install Packages

Install the required libraries for running Qwen (with GPU support) and setting up the API.

In [ ]:
!pip install transformers accelerate bitsandbytes
!pip install fastapi uvicorn pyngrok nest_asyncio httpx

## Step 2: Load Qwen Model

Initialize the tokenizer and load Qwen 3 8B Instruct using 4-bit quantization to fit on Colab's T4 GPU.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    load_in_4bit=True
)

## Step 3: Create FastAPI Server

Define endpoints to process the deterministic workspace data and output enhanced assessments.

In [ ]:
from fastapi import FastAPI

app = FastAPI(title="Qwen Augmentation Server")

@app.post("/generate-assessment")
async def generate_assessment(data: dict):
    goal = data.get("goal", "")
    assessment = data.get("assessment", "")
    
    prompt = f"""Improve this intelligence assessment. Fix language, scenarios, and alternative explanations while maintaining empirical metrics.

Goal:
{goal}

Assessment:
{assessment}
"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    input_tokens = inputs.input_ids.shape[1]
    
    outputs = model.generate(**inputs, max_new_tokens=300)
    output_tokens = outputs.shape[1] - input_tokens
    
    text = tokenizer.decode(outputs[0][input_tokens:], skip_special_tokens=True)
    
    return {
        "enhanced_assessment": text.strip(),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens
    }

## Step 4: Expose Server with ngrok

Establish the ngrok tunnel. Replace `YOUR_AUTHTOKEN` with your free ngrok token if prompted or required.

In [ ]:
from pyngrok import ngrok

# Optional: ngrok.set_auth_token("YOUR_AUTHTOKEN")
public_url = ngrok.connect(8000)
print("Public Endpoint:", public_url)

## Step 5: Start FastAPI Server

Run uvicorn. Note down the public URL output from the previous cell and add it to your local `.env` file.

In [ ]:
import nest_asyncio
import uvicorn

nest_asyncio.apply()
uvicorn.run(app, host="0.0.0.0", port=8000)